# Classical Retrieval: BM25, Dense, Hybrid (RRF), SPLADE

Этот ноутбук реализует классический retrieval pipeline для Enterprise RAG:

1. **BM25** — лексический baseline с **стеммизацией** (Porter Stemmer)
2. **Dense** — семантический retrieval (sentence-transformers + FAISS)
3. **Hybrid BM25+Dense** — Reciprocal Rank Fusion (RRF)
4. **SPLADE** — sparse neural retrieval (transformers)
5. **Hybrid BM25+Dense+SPLADE** — трёхкомпонентный RRF

**Ключевая особенность**: поиск ведётся **почанково** — каждый чанк является
независимой единицей индексации и поиска. Релевантность определяется на уровне
чанка, ground truth агрегируется с уровня документа до уровня чанков.

## 0. Setup & Imports

In [ ]:
#!pip install -q rank-bm25 faiss-cpu sentence-transformers transformers torch nltk tqdm

In [1]:
import pandas as pd
import numpy as np
import time
import json
import re
from typing import List, Dict, Tuple, Optional
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm

# BM25
from rank_bm25 import BM25Okapi

# Dense retrieval
import faiss
from sentence_transformers import SentenceTransformer

# SPLADE
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

# Stemming for BM25
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import nltk

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

print("Libraries loaded successfully")

I0000 00:00:1783869959.395745   15968 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783869959.436418   15968 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783869960.543074   15968 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783869960.543336   15968 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Libraries loaded successfully


## 1. Load Data

Загружаем чанки (fixed_v1) и вопросы.

**Важно**: для retrieval мы работаем на уровне **чанков**, не документов.
Каждый чанк — независимая единица. Ground truth маппится с документов на чанки.

In [2]:
# Загружаем чанки
df_chunks = pd.read_parquet("chunks_fixed_v1.parquet")
print(f"Chunks: {len(df_chunks)} rows, {df_chunks['doc_id'].nunique()} unique docs")

# Загружаем вопросы
df_questions = pd.read_parquet("questions_for_sample.parquet")
print(f"Questions: {len(df_questions)}")

# Смотрим на структуру
print("\nChunk columns:", list(df_chunks.columns))
print("\nQuestion columns:", list(df_questions.columns))
print("\nSample chunk:")
print(df_chunks.iloc[0]['chunk_text'][:200] + "...")

Chunks: 570005 rows, 100000 unique docs
Questions: 138

Chunk columns: ['chunk_id', 'doc_id', 'source_type', 'chunk_idx', 'chunk_text', 'n_tokens', 'strategy']

Question columns: ['question_id', 'question_type', 'source_types', 'question', 'expected_doc_ids', 'gold_answer', 'answer_facts']

Sample chunk:
["From: Alyssa Young <alyssa@edgetech.io>\nTo: Mateo Alvarez <mateo_alvarez@redwood.com>\nCc: procure@edgetech.io <procure@edgetech.io>\nDate: Sat, 17 Oct 2026 09:12:00 -0700\nSubject: Onboarding cred...


In [ ]:
# Подготовка ground truth на уровне чанков
# Для каждого вопроса: expected_doc_ids -> все чанки этих документов

def prepare_ground_truth_chunks(df_questions, df_chunks):
    """
    Создаёт словарь question_id -> set(expected_chunk_ids)
    Все чанки из expected_doc_ids считаются релевантными.
    """
    # Создаём маппинг doc_id -> set(chunk_ids)
    doc_to_chunks = defaultdict(set)
    for _, row in tqdm(df_chunks.iterrows(), total=len(df_chunks), desc="Mapping doc→chunks"):
        doc_to_chunks[row['doc_id']].add(row['chunk_id'])
    
    gt = {}
    for _, row in tqdm(df_questions.iterrows(), total=len(df_questions), desc="Building ground truth"):
        qid = row['question_id']
        expected_chunks = set()
        for doc_id in row['expected_doc_ids']:
            expected_chunks.update(doc_to_chunks.get(doc_id, set()))
        gt[qid] = expected_chunks
    
    return gt

ground_truth = prepare_ground_truth_chunks(df_questions, df_chunks)

# Создаём список вопросов для evaluation
queries = []
for _, row in tqdm(df_questions.iterrows(), total=len(df_questions), desc="Preparing queries"):
    queries.append({
        'qid': row['question_id'],
        'text': row['question'],
        'expected_chunks': ground_truth[row['question_id']]
    })

print(f"Queries for evaluation: {len(queries)}")
print(f"Sample query: {queries[0]['text'][:80]}...")
print(f"Expected chunks: {len(queries[0]['expected_chunks'])}")

# Статистика по ground truth
gt_sizes = [len(chunks) for chunks in ground_truth.values()]
print(f"\nGround truth stats (chunks per question):")
print(f"  Mean: {np.mean(gt_sizes):.1f}")
print(f"  Median: {np.median(gt_sizes):.1f}")
print(f"  Min: {min(gt_sizes)}")
print(f"  Max: {max(gt_sizes)}")

Preparing queries: 100%|██████████| 138/138 [00:00<00:00, 41659.27it/s]

Queries for evaluation: 138
Sample query: What is the name of the new metric added so SRE can track when server-side strea...
Expected chunks: 4

Ground truth stats (chunks per question):
  Mean: 7.9
  Median: 6.0
  Min: 1
  Max: 42


## 2. Evaluation Metrics (Chunk-level)

In [35]:
def compute_metrics_chunk_level(retrieved_lists: Dict[str, List[str]], 
                                ground_truth: Dict[str, set], 
                                k_values: List[int] = [1, 5, 10, 100, 1000]) -> pd.DataFrame:
    """
    Вычисляет метрики для retrieval результатов на уровне чанков.
    
    Args:
        retrieved_lists: {qid -> [chunk_id_1, chunk_id_2, ...]} — отранжированные списки чанков
        ground_truth: {qid -> {chunk_id_1, chunk_id_2, ...}} — релевантные чанки
        k_values: список k для @k метрик
    """
    results = []
    
    for k in k_values:
        recalls = []
        ndcgs = []
        mrrs = []
        
        for qid, expected in tqdm(ground_truth.items(), desc=f"Metrics@k={k}", leave=False):
            if qid not in retrieved_lists:
                continue
            
            retrieved = retrieved_lists[qid][:k]
            n_expected = len(expected)
            
            if n_expected == 0:
                continue
            
            # Recall@k
            n_found = len(set(retrieved) & expected)
            recall = n_found / n_expected
            recalls.append(recall)
            
            # NDCG@k
            dcg = 0.0
            for i, chunk_id in enumerate(retrieved):
                rel = 1.0 if chunk_id in expected else 0.0
                dcg += rel / np.log2(i + 2)
            
            ideal_rels = [1.0] * min(n_expected, k)
            idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_rels))
            ndcg = dcg / idcg if idcg > 0 else 0.0
            ndcgs.append(ndcg)
            
            # MRR@k
            rr = 0.0
            for i, chunk_id in enumerate(retrieved):
                if chunk_id in expected:
                    rr = 1.0 / (i + 1)
                    break
            mrrs.append(rr)
        
        results.append({
            'k': k,
            'Recall@k': np.mean(recalls) if recalls else 0.0,
            'NDCG@k': np.mean(ndcgs) if ndcgs else 0.0,
            'MRR@k': np.mean(mrrs) if mrrs else 0.0,
            'n_queries': len(recalls),
        })
    
    return pd.DataFrame(results)


def print_metrics_table(results_df: pd.DataFrame, title: str = ""):
    """Красивый вывод метрик"""
    if title:
        print(f"\n{'='*60}")
        print(f"  {title}")
        print(f"{'='*60}")
    print(results_df.round(4).to_string(index=False))
    print()

print("Metrics functions defined (chunk-level)")


chunk_to_doc = dict(zip(df_chunks['chunk_id'], df_chunks['doc_id']))
expected_doc_ids_by_qid = {row['question_id']: set(row['expected_doc_ids'])
                            for _, row in df_questions.iterrows()}


def compute_metrics_doc_level(retrieved_lists: Dict[str, List[str]],
                               chunk_to_doc: Dict[str, str],
                               expected_doc_ids: Dict[str, set],
                               k_values: List[int] = [1, 5, 10, 100, 1000]) -> pd.DataFrame:
    """
    Метрики на уровне документа: чанк считается 'попаданием',
    если его doc_id входит в expected_doc_ids вопроса.
    Дедупликация: несколько чанков одного релевантного doc_id в топе
    считаются за одно попадание (используется первое вхождение).
    """
    results = []

    for k in k_values:
        recalls, ndcgs, mrrs = [], [], []

        for qid, expected_docs in tqdm(expected_doc_ids.items(), desc=f"Doc metrics@k={k}", leave=False):
            if qid not in retrieved_lists or not expected_docs:
                continue

            retrieved_chunks = retrieved_lists[qid][:k]
            # Схлопываем чанки в doc_id, сохраняя порядок первого вхождения
            seen_docs = []
            seen_set = set()
            for chunk_id in retrieved_chunks:
                doc_id = chunk_to_doc.get(chunk_id)
                if doc_id is not None and doc_id not in seen_set:
                    seen_set.add(doc_id)
                    seen_docs.append(doc_id)

            n_expected = len(expected_docs)

            # Recall@k (доля найденных ожидаемых документов)
            n_found = len(seen_set & expected_docs)
            recalls.append(n_found / n_expected)

            # NDCG@k на уровне документов
            dcg = 0.0
            for i, doc_id in enumerate(seen_docs):
                rel = 1.0 if doc_id in expected_docs else 0.0
                dcg += rel / np.log2(i + 2)
            ideal_rels = [1.0] * min(n_expected, k)
            idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_rels))
            ndcgs.append(dcg / idcg if idcg > 0 else 0.0)

            # MRR@k
            rr = 0.0
            for i, doc_id in enumerate(seen_docs):
                if doc_id in expected_docs:
                    rr = 1.0 / (i + 1)
                    break
            mrrs.append(rr)

        results.append({
            'k': k,
            'Recall@k': np.mean(recalls) if recalls else 0.0,
            'NDCG@k': np.mean(ndcgs) if ndcgs else 0.0,
            'MRR@k': np.mean(mrrs) if mrrs else 0.0,
            'n_queries': len(recalls),
        })

    return pd.DataFrame(results)

Metrics functions defined (chunk-level)


## 3. BM25 — Lexical Baseline with Stemming

Используем rank-bm25 с **стеммизацией** (Porter Stemmer) и удалением стоп-слов.

**Преимущества стеммизации:**
- Сводит разные формы слова к общему корню (running -> run, better -> better)
- Улучшает recall за счёт матчинга морфологических вариантов
- Уменьшает размер словаря

**Недостатки:**
- Может агрессивно стеммизировать (university -> univers)
- Не учитывает контекст (stemmer — эвристический)

In [29]:
!pip install -q bm25s[full]

In [36]:
import bm25s
import Stemmer

stemmer = Stemmer.Stemmer('russian')

print("Tokenizing chunks for bm25s...")
t0 = time.time()

chunk_texts_list = df_chunks['chunk_text'].tolist()
bm25s_chunk_ids = df_chunks['chunk_id'].tolist()  # порядок тот же, что в df_chunks — можно переиспользовать старый bm25_chunk_ids, если он строился из того же df_chunks в том же порядке

corpus_tokens = bm25s.tokenize(chunk_texts_list, stopwords="ru", stemmer=stemmer)

bm25 = bm25s.BM25()
bm25.index(corpus_tokens)

t1 = time.time()
print(f"bm25s index built in {t1-t0:.1f}s")

# Сохраняем в новый, отдельный кэш — не перезаписывай старый bm25_cache/
bm25.save("bm25s_cache/index")
import pickle, os
os.makedirs("bm25s_cache", exist_ok=True)
with open("bm25s_cache/chunk_ids.pkl", "wb") as f:
    pickle.dump(bm25s_chunk_ids, f)

Tokenizing chunks for bm25s...


Split strings:   0%|          | 0/570005 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/570005 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/570005 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/570005 [00:00<?, ?it/s]

bm25s index built in 69.1s


In [38]:
# восстановление в новой сессии
bm25 = bm25s.BM25.load("bm25s_cache/index")
with open("bm25s_cache/chunk_ids.pkl", "rb") as f:
    bm25s_chunk_ids = pickle.load(f)

# поиск (батчево, все запросы разом)
query_tokens = bm25s.tokenize([q['text'] for q in queries], stopwords="ru", stemmer=stemmer)
doc_indices, scores = bm25.retrieve(query_tokens, k=1000)

bm25s_results = {}
for i, query in enumerate(queries):
    bm25s_results[query['qid']] = [bm25s_chunk_ids[idx] for idx in doc_indices[i]]

Split strings:   0%|          | 0/138 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/138 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/138 [00:00<?, ?it/s]

In [39]:
# Поиск BM25 (на уровне чанков)
def search_bm25_chunks(queries: List[Dict], bm25, chunk_ids: List[str], stemmer, top_k: int = 1000) -> Dict[str, List[str]]:
    """Поиск по BM25 (bm25s) — возвращает чанки"""
    query_texts = [q['text'] for q in queries]
    query_tokens = bm25s.tokenize(query_texts, stopwords="ru", stemmer=stemmer)

    doc_indices, scores = bm25.retrieve(query_tokens, k=top_k)

    results = {}
    for i, query in enumerate(queries):
        results[query['qid']] = [chunk_ids[idx] for idx in doc_indices[i]]
    return results

print("Searching with BM25 (chunk-level)...")
t0 = time.time()
bm25_results = search_bm25_chunks(queries, bm25, bm25s_chunk_ids, stemmer, top_k=1000)
t1 = time.time()

print(f"BM25 search completed in {t1-t0:.1f}s ({(t1-t0)/len(queries):.3f}s per query)")

# Evaluation
bm25_metrics = compute_metrics_doc_level(bm25_results, chunk_to_doc, expected_doc_ids_by_qid)
print_metrics_table(bm25_metrics, "BM25 with Stemming (Doc-level) Results")

Searching with BM25 (chunk-level)...


Split strings:   0%|          | 0/138 [00:00<?, ?it/s]

Stem Tokens:   0%|          | 0/138 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/138 [00:00<?, ?it/s]

BM25 search completed in 1.4s (0.010s per query)


...

Doc metrics@k=5:   0%|          | 0/138 [00:00<?, ?it/s]

Doc metrics@k=10:   0%|          | 0/138 [00:00<?, ?it/s]

Doc metrics@k=100:   0%|          | 0/138 [00:00<?, ?it/s]

Doc metrics@k=1000:   0%|          | 0/138 [00:00<?, ?it/s]


  BM25 with Stemming (Doc-level) Results
   k  Recall@k  NDCG@k  MRR@k  n_queries
   1    0.3875  0.5000 0.5000        138
   5    0.5338  0.5054 0.5886        138
  10    0.5689  0.5127 0.5948        138
 100    0.6746  0.5371 0.6008        138
1000    0.7208  0.5440 0.6012        138



In [34]:
import pickle
import os

# Сохраняем объект BM25 и список chunk_id
os.makedirs('bm25_cache', exist_ok=True)

with open('bm25_cache/bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25, f)

with open('bm25_cache/bm25_chunk_ids.pkl', 'wb') as f:
    pickle.dump(bm25_chunk_ids, f)

# (Опционально) сохраняем токенизированный корпус, если понадобится
with open('bm25_cache/bm25_corpus.pkl', 'wb') as f:
    pickle.dump(bm25_corpus, f)

print("BM25 index, chunk_ids and corpus saved.")

BM25 index, chunk_ids and corpus saved.


## 4. Dense Retrieval (Chunk-level)

Используем sentence-transformers для получения dense embeddings **каждого чанка**.

**Модель:** `intfloat/e5-small-v2` — быстрая, компактная (33M параметров).
Префикс `passage:` для чанков, `query:` для запросов.

**Индекс:** FAISS IndexFlatIP (Inner Product) — точный поиск.

**Важно**: никакой агрегации чанков в документы — каждый чанк индексируется отдельно.

In [7]:
import os
os.environ["OMP_NUM_THREADS"] = "6"

from pathlib import Path
from optimum.onnxruntime import ORTModelForFeatureExtraction
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from optimum.onnxruntime import ORTQuantizer
from transformers import AutoTokenizer

DENSE_MODEL = "intfloat/e5-small-v2"
ONNX_DIR = Path("onnx_e5_small")
QUANT_DIR = Path("onnx_e5_small_int8")

tokenizer = AutoTokenizer.from_pretrained(DENSE_MODEL)

if not (QUANT_DIR / "model_quantized.onnx").exists():
    # 1. Экспорт в ONNX
    ort_model = ORTModelForFeatureExtraction.from_pretrained(DENSE_MODEL, export=True)
    ort_model.save_pretrained(ONNX_DIR)
    tokenizer.save_pretrained(ONNX_DIR)

    # 2. Dynamic INT8 quantization (avx512_vnni — под твой Meteor Lake, если не встанет — avx2)
    quantizer = ORTQuantizer.from_pretrained(ONNX_DIR)
    qconfig = AutoQuantizationConfig.avx512_vnni(is_static=False, per_channel=False)
    quantizer.quantize(save_dir=QUANT_DIR, quantization_config=qconfig)
    print("Quantized model saved")

In [8]:
import time
import numpy as np
import onnxruntime as ort
from tqdm.auto import tqdm
from optimum.onnxruntime import ORTModelForFeatureExtraction

sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = 6
sess_options.inter_op_num_threads = 1
sess_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

sess_options.enable_mem_pattern = True
sess_options.enable_cpu_mem_arena = True
sess_options.add_session_config_entry("session.intra_op.allow_spinning", "1")

t0 = time.time()
model = ORTModelForFeatureExtraction.from_pretrained(
    QUANT_DIR,
    file_name="model_quantized.onnx",
    session_options=sess_options,
    provider="CPUExecutionProvider",
)
print(f"Model loaded in {time.time()-t0:.1f}s")


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask[..., None].astype(np.float32)
    summed = (last_hidden_state * mask).sum(axis=1)
    counts = np.clip(mask.sum(axis=1), a_min=1e-9, a_max=None)
    return summed / counts


def encode(texts, batch_size=64, max_length=256, show_progress_bar=True):
    # Сортируем по длине (в словах, как прокси токенов) — батчи получаются однородными
    order = np.argsort([len(t) for t in texts])
    sorted_texts = [texts[i] for i in order]

    embs = []
    n_batches = (len(sorted_texts) + batch_size - 1) // batch_size
    iterator = range(0, len(sorted_texts), batch_size)
    if show_progress_bar:
        iterator = tqdm(iterator, total=n_batches, desc="Encoding", unit="batch")

    for i in iterator:
        batch = sorted_texts[i:i + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True,
                         max_length=max_length, return_tensors="np")
        out = model(**enc)
        pooled = mean_pool(out.last_hidden_state, enc["attention_mask"])
        pooled = pooled / np.linalg.norm(pooled, axis=1, keepdims=True)
        embs.append(pooled.astype("float32"))

    embs = np.concatenate(embs, axis=0)
    # Возвращаем в исходный порядок
    inverse_order = np.argsort(order)
    return embs[inverse_order]

Model loaded in 0.2s


In [9]:
print("Computing CHUNK embeddings...")
t0 = time.time()

chunk_texts = [f"passage: {text}" for text in df_chunks['chunk_text'].tolist()]
dense_chunk_ids = df_chunks['chunk_id'].tolist()

chunk_embeddings = encode(chunk_texts, batch_size=64, max_length=256)

t1 = time.time()
print(f"\nEmbeddings computed in {t1-t0:.1f}s")
print(f"Shape: {chunk_embeddings.shape}")
print(f"Memory: {chunk_embeddings.nbytes / 1024 / 1024:.1f} MB")

Computing CHUNK embeddings...


Encoding:   0%|          | 0/8907 [00:00<?, ?batch/s]


Embeddings computed in 14342.3s
Shape: (570005, 384)
Memory: 835.0 MB


In [41]:
# Создаём FAISS индекс
print("Building FAISS index...")
t0 = time.time()

dim = chunk_embeddings.shape[1]
dense_index = faiss.IndexFlatIP(dim)  # Inner Product = Cosine для нормализованных векторов
dense_index.add(chunk_embeddings)

t1 = time.time()
print(f"FAISS index built in {t1-t0:.1f}s")
print(f"Index size: {dense_index.ntotal} vectors")

Building FAISS index...
FAISS index built in 0.3s
Index size: 570005 vectors


In [45]:
# Поиск Dense (на уровне чанков)
def search_dense_chunks(queries: List[Dict], model, index, chunk_ids: List[str], top_k: int = 1000) -> Dict[str, List[str]]:
    """Поиск по dense embeddings — возвращает чанки"""
    results = {}
    
    # Кодируем запросы батчем с прогресс-баром
    query_texts = [f"query: {q['text']}" for q in queries]
    print(f"Encoding {len(query_texts)} queries...")
    query_embeddings = encode(query_texts, batch_size=16)
    
    # Нормализуем
    query_embeddings = query_embeddings / np.linalg.norm(query_embeddings, axis=1, keepdims=True)
    query_embeddings = query_embeddings.astype('float32')
    
    # Поиск в FAISS
    print("Searching in FAISS index...")
    scores, indices = index.search(query_embeddings, top_k)
    
    for i, query in enumerate(tqdm(queries, desc="Building results")):
        results[query['qid']] = [chunk_ids[idx] for idx in indices[i]]
    
    return results

print("Searching with Dense retrieval (chunk-level)...")
t0 = time.time()
dense_results = search_dense_chunks(queries, model, dense_index, dense_chunk_ids, top_k=1000)
t1 = time.time()

print(f"Dense search completed in {t1-t0:.1f}s ({(t1-t0)/len(queries):.3f}s per query)")

# Evaluation
dense_metrics = compute_metrics_doc_level(dense_results, chunk_to_doc, expected_doc_ids_by_qid)
print_metrics_table(dense_metrics, "Dense Retrieval e5-small (Chunk-level) Results")

Searching with Dense retrieval (chunk-level)...
Encoding 138 queries...


Encoding:   0%|          | 0/9 [00:00<?, ?batch/s]

Searching in FAISS index...


Building results:   0%|          | 0/138 [00:00<?, ?it/s]

Dense search completed in 1.9s (0.014s per query)


...

Doc metrics@k=5:   0%|          | 0/138 [00:00<?, ?it/s]

Doc metrics@k=10:   0%|          | 0/138 [00:00<?, ?it/s]

Doc metrics@k=100:   0%|          | 0/138 [00:00<?, ?it/s]

Doc metrics@k=1000:   0%|          | 0/138 [00:00<?, ?it/s]


  Dense Retrieval e5-small (Chunk-level) Results
   k  Recall@k  NDCG@k  MRR@k  n_queries
   1    0.2414  0.3188 0.3188        138
   5    0.3977  0.3554 0.4115        138
  10    0.4501  0.3706 0.4196        138
 100    0.5526  0.3942 0.4251        138
1000    0.6674  0.4094 0.4257        138



In [28]:
import faiss
import numpy as np
import pickle
import os

os.makedirs('dense', exist_ok=True)

# Эмбеддинги чанков (numpy array, float32, нормализованные)
np.save('dense/chunk_embeddings.npy', chunk_embeddings)

# FAISS индекс
faiss.write_index(dense_index, 'dense/faiss_index.bin')

# chunk_id → порядок в индексе (без этого эмбеддинги бесполезны — нет маппинга индекс→chunk_id)
with open('dense/dense_chunk_ids.pkl', 'wb') as f:
    pickle.dump(dense_chunk_ids, f)

print(f"Saved embeddings: {chunk_embeddings.shape}, {chunk_embeddings.nbytes / 1024**2:.1f} MB")
print(f"Saved FAISS index: {dense_index.ntotal} vectors")
print(f"Saved chunk_ids mapping: {len(dense_chunk_ids)} ids")

Saved embeddings: (570005, 384), 835.0 MB
Saved FAISS index: 570005 vectors
Saved chunk_ids mapping: 570005 ids


## 5. Hybrid BM25 + Dense (RRF) — Chunk-level

Reciprocal Rank Fusion (RRF) объединяет ранжирования на уровне чанков.

In [47]:
def reciprocal_rank_fusion(rankings: List[List[str]], k: int = 60) -> List[Tuple[str, float]]:
    """
    Объединяет несколько ранжирований через RRF.
    
    Args:
        rankings: список списков chunk_id (каждый отсортирован по релевантности)
        k: константа RRF (обычно 60)
    
    Returns:
        Список (chunk_id, rrf_score), отсортированный по убыванию score
    """
    scores = defaultdict(float)
    
    for ranking in rankings:
        for rank, chunk_id in enumerate(ranking):
            scores[chunk_id] += 1.0 / (k + rank + 1)
    
    sorted_results = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_results


def search_hybrid_chunk_level(bm25_results: Dict, dense_results: Dict, k: int = 60, top_k: int = 1000) -> Dict[str, List[str]]:
    """Гибридный поиск через RRF на уровне чанков"""
    results = {}
    for qid in tqdm(bm25_results, desc="Hybrid BM25+Dense RRF"):
        if qid not in dense_results:
            continue
        rrf_scores = reciprocal_rank_fusion(
            [bm25_results[qid], dense_results[qid]],
            k=k
        )
        results[qid] = [chunk_id for chunk_id, _ in rrf_scores[:top_k]]
    return results

print("Computing Hybrid BM25+Dense (RRF, chunk-level)...")
t0 = time.time()
hybrid_results = search_hybrid_chunk_level(bm25_results, dense_results, k=60, top_k=1000)
t1 = time.time()

print(f"Hybrid search completed in {t1-t0:.1f}s")

# Evaluation
hybrid_metrics = compute_metrics_doc_level(hybrid_results, chunk_to_doc, expected_doc_ids_by_qid )
print_metrics_table(hybrid_metrics, "Hybrid BM25+Dense RRF (Chunk-level) Results")

Computing Hybrid BM25+Dense (RRF, chunk-level)...


Hybrid BM25+Dense RRF:   0%|          | 0/138 [00:00<?, ?it/s]

Hybrid search completed in 0.1s


Doc metrics@k=1:   0%|          | 0/138 [00:00<?, ?it/s]

Doc metrics@k=5:   0%|          | 0/138 [00:00<?, ?it/s]

Doc metrics@k=10:   0%|          | 0/138 [00:00<?, ?it/s]

Doc metrics@k=100:   0%|          | 0/138 [00:00<?, ?it/s]

Doc metrics@k=1000:   0%|          | 0/138 [00:00<?, ?it/s]


  Hybrid BM25+Dense RRF (Chunk-level) Results
   k  Recall@k  NDCG@k  MRR@k  n_queries
   1    0.4042  0.5290 0.5290        138
   5    0.5141  0.5009 0.5944        138
  10    0.5653  0.5151 0.6045        138
 100    0.6674  0.5398 0.6111        138
1000    0.7198  0.5476 0.6115        138



## 6. SPLADE — Sparse Neural Retrieval (Chunk-level)

SPLADE генерирует разреженные вектора для каждого чанка отдельно.

In [22]:
# Загружаем модель для SPLADE
SPLADE_MODEL = "naver/splade-cocondenser-ensembledistil"

print(f"Loading SPLADE model: {SPLADE_MODEL}...")
t0 = time.time()

splade_tokenizer = AutoTokenizer.from_pretrained(SPLADE_MODEL)
splade_model = AutoModelForMaskedLM.from_pretrained(SPLADE_MODEL)
splade_model.eval()

if torch.cuda.is_available():
    splade_model = splade_model.cuda()
    device = 'cuda'
else:
    device = 'cpu'

t1 = time.time()
print(f"Model loaded in {t1-t0:.1f}s (device: {device})")

Loading SPLADE model: naver/splade-cocondenser-ensembledistil...
Model loaded in 1.3s (device: cpu)


In [24]:
import os
os.environ["OMP_NUM_THREADS"] = "16"

import torch
torch.set_num_threads(16)

# Dynamic INT8 quantization SPLADE-модели (тот же трюк, что для dense)
splade_model_q = torch.quantization.quantize_dynamic(
    splade_model, {torch.nn.Linear}, dtype=torch.qint8
)
splade_model_q.eval()


def encode_splade(texts, tokenizer, model, batch_size=32, max_length=256, show_progress=True):
    from scipy.sparse import csr_matrix, vstack

    vocab_size = tokenizer.vocab_size

    # Сортировка по длине — минимизирует padding в батчах
    order = np.argsort([len(t) for t in texts])
    sorted_texts = [texts[i] for i in order]

    chunks = []
    iterator = range(0, len(sorted_texts), batch_size)
    if show_progress:
        iterator = tqdm(iterator, total=(len(sorted_texts) + batch_size - 1) // batch_size,
                         desc="SPLADE encoding")

    with torch.no_grad():
        for i in iterator:
            batch = sorted_texts[i:i + batch_size]
            inputs = tokenizer(batch, return_tensors='pt', padding=True,
                                truncation=True, max_length=max_length)

            outputs = model(**inputs).logits
            max_over_seq = torch.max(outputs, dim=1).values
            weights = torch.log1p(torch.relu(max_over_seq))  # log1p стабильнее log(1+x)

            weights = weights.numpy()

            all_indices, all_values, all_indptr = [], [], [0]
            for w in weights:
                mask = w > 0.01
                idx = np.where(mask)[0]
                all_indices.extend(idx.tolist())
                all_values.extend(w[mask].tolist())
                all_indptr.append(len(all_indices))

            chunks.append(csr_matrix((all_values, all_indices, all_indptr),
                                      shape=(len(batch), vocab_size)))

    sparse_matrix = vstack(chunks)

    # Возвращаем в исходный порядок
    inverse_order = np.argsort(order)
    return sparse_matrix[inverse_order]

In [26]:
# Эмбеддинги ЧАНКОВ через SPLADE
print("Computing SPLADE chunk embeddings...")
print("(This may take a while for 570K chunks)")
t0 = time.time()

splade_chunk_texts = df_chunks['chunk_text'].tolist()
splade_chunk_ids = df_chunks['chunk_id'].tolist()

splade_chunk_embeddings = encode_splade(
    splade_chunk_texts, splade_tokenizer, splade_model_q,
    batch_size=32, max_length=256,
)

t1 = time.time()
print(f"\nSPLADE embeddings computed in {t1-t0:.1f}s")
print(f"Shape: {splade_chunk_embeddings.shape}")
print(f"Sparsity: {1 - splade_chunk_embeddings.nnz / (splade_chunk_embeddings.shape[0] * splade_chunk_embeddings.shape[1]):.4f}")
print(f"Avg non-zero per chunk: {splade_chunk_embeddings.nnz / splade_chunk_embeddings.shape[0]:.0f}")

Computing SPLADE chunk embeddings...
(This may take a while for 570K chunks)


SPLADE encoding:   0%|          | 0/17813 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Поиск SPLADE (на уровне чанков)
def search_splade_chunks(queries, tokenizer, model, chunk_embeddings, chunk_ids,
                          top_k: int = 1000):
    results = {}
    query_texts = [q['text'] for q in queries]

    query_embeddings = encode_splade(query_texts, tokenizer, model,
                                      batch_size=32, show_progress=False)

    similarities = query_embeddings @ chunk_embeddings.T  # sparse x sparse

    for i, query in enumerate(tqdm(queries, desc="SPLADE search")):
        row = similarities[i].toarray().flatten()
        # argpartition вместо argsort — O(n) вместо O(n log n)
        top_k_idx = np.argpartition(row, -top_k)[-top_k:]
        top_k_idx = top_k_idx[np.argsort(row[top_k_idx])[::-1]]
        results[query['qid']] = [chunk_ids[idx] for idx in top_k_idx]

    return results

print("Searching with SPLADE (chunk-level)...")
t0 = time.time()
splade_results = search_splade_chunks(
    queries, splade_tokenizer, splade_model, 
    splade_chunk_embeddings, splade_chunk_ids, 
    device=device, top_k=1000
)
t1 = time.time()

print(f"\nSPLADE search completed in {t1-t0:.1f}s ({(t1-t0)/len(queries):.3f}s per query)")

# Evaluation
splade_metrics = compute_metrics_chunk_level(splade_results, ground_truth)
print_metrics_table(splade_metrics, "SPLADE (Chunk-level) Results")

## 7. Triple Hybrid: BM25 + Dense + SPLADE (RRF) — Chunk-level

In [ ]:
print("Computing Triple Hybrid BM25+Dense+SPLADE (RRF, chunk-level)...")
t0 = time.time()

triple_results = {}
for qid in tqdm(bm25_results, desc="Triple Hybrid RRF"):
    if qid not in dense_results or qid not in splade_results:
        continue
    rrf_scores = reciprocal_rank_fusion(
        [bm25_results[qid], dense_results[qid], splade_results[qid]],
        k=60
    )
    triple_results[qid] = [chunk_id for chunk_id, _ in rrf_scores[:1000]]

t1 = time.time()
print(f"Triple hybrid completed in {t1-t0:.1f}s")

# Evaluation
triple_metrics = compute_metrics_doc_level(triple_results, ground_truth, expected_doc_ids_by_qid )
print_metrics_table(triple_metrics, "Triple Hybrid BM25+Dense+SPLADE (Chunk-level) Results")

## 8. Comparative Analysis

In [ ]:
# Собираем все метрики в одну таблицу
all_metrics = []

for metrics_df, name in [
    (bm25_metrics, "BM25 + Stemming"),
    (dense_metrics, "Dense (e5-small)"),
    (hybrid_metrics, "Hybrid BM25+Dense"),
    (splade_metrics, "SPLADE"),
    (triple_metrics, "Triple Hybrid"),
]:
    for _, row in metrics_df.iterrows():
        all_metrics.append({
            'Method': name,
            'k': int(row['k']),
            'Recall@k': row['Recall@k'],
            'NDCG@k': row['NDCG@k'],
            'MRR@k': row['MRR@k'],
        })

comparison_df = pd.DataFrame(all_metrics)

# Pivot для удобного сравнения
for metric in ['Recall@k', 'NDCG@k', 'MRR@k']:
    print(f"\n{'='*70}")
    print(f"  {metric}")
    print(f"{'='*70}")
    pivot = comparison_df.pivot(index='Method', columns='k', values=metric)
    print(pivot.round(4).to_string())
    print()

In [ ]:
# Визуализация сравнения
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

methods = ['BM25 + Stemming', 'Dense (e5-small)', 'Hybrid BM25+Dense', 'SPLADE', 'Triple Hybrid']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for ax, metric in zip(axes, ['Recall@k', 'NDCG@k', 'MRR@k']):
    for method, color in zip(methods, colors):
        data = comparison_df[comparison_df['Method'] == method]
        ax.plot(data['k'], data[metric], marker='o', label=method, color=color, linewidth=2)
    
    ax.set_xlabel('k')
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.set_xscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('retrieval_comparison_chunks.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nVisualization saved to retrieval_comparison_chunks.png")

## 9. Save Results

Сохраняем результаты для дальнейшего использования (reranking, generation).

**Важно**: результаты содержат chunk_id, не doc_id. Для reranking нужно будет
маппить chunk_id обратно на тексты через df_chunks.

In [ ]:
import pickle
import os

# Создаём директорию для результатов
os.makedirs('retrieval_results', exist_ok=True)

# Сохраняем все результаты (chunk-level)
results_to_save = {
    'bm25_stemming_chunks': bm25_results,
    'dense_chunks': dense_results,
    'hybrid_bm25_dense_chunks': hybrid_results,
    'splade_chunks': splade_results,
    'triple_hybrid_chunks': triple_results,
}

for name, results in results_to_save.items():
    filepath = f'retrieval_results/{name}_top1000.pkl'
    with open(filepath, 'wb') as f:
        pickle.dump(results, f)
    print(f"Saved: {filepath}")

# Сохраняем метрики
metrics_to_save = {
    'bm25_stemming': bm25_metrics,
    'dense': dense_metrics,
    'hybrid_bm25_dense': hybrid_metrics,
    'splade': splade_metrics,
    'triple_hybrid': triple_metrics,
}

for name, metrics in metrics_to_save.items():
    filepath = f'retrieval_results/{name}_metrics.csv'
    metrics.to_csv(filepath, index=False)
    print(f"Saved: {filepath}")

# Сохраняем сводную таблицу
comparison_df.to_csv('retrieval_results/comparison_all_methods_chunks.csv', index=False)
print(f"\nSaved: retrieval_results/comparison_all_methods_chunks.csv")

print("\nAll results saved successfully!")

## 10. Summary & Next Steps

### Что сделано:
1. ✅ **BM25 + Stemming** — лексический baseline со стеммизацией Porter
2. ✅ **Dense (e5-small)** — семантический retrieval на уровне чанков
3. ✅ **Hybrid BM25+Dense** — RRF fusion на уровне чанков
4. ✅ **SPLADE** — sparse neural retrieval на уровне чанков
5. ✅ **Triple Hybrid** — все три метода через RRF на уровне чанков

### Ключевые изменения от предыдущей версии:
- **Поиск почанково**: каждый чанк — независимая единица
- **Стеммизация**: Porter Stemmer + удаление стоп-слов для BM25
- **Ground truth**: маппинг с doc_id на chunk_id

### Следующие шаги (reranking + generation):
1. **Reranking** — cross-encoder на top-100 чанков лучшего метода
2. **Aggregation** — агрегация чанков в документы для финальной выдачи
3. **Generation** — RAG с top-10 документов через LLM
4. **Evaluation** — LLM-as-a-judge для оценки ответов